# TimeSformer Inference Notebook

Run inference on driving distraction videos using a fine-tuned TimeSformer-HR checkpoint.

### What you need
The `best_model/` folder produced by `Finetuning.py` must be available. It must contain:
- `model.safetensors` — fine-tuned weights (**required**)
- `config.json` — model architecture + label map (**required**)
- `preprocessor_config.json` — feature-extractor config (**optional** — if missing, it is loaded automatically from `facebook/timesformer-hr-finetuned-k400` on Hugging Face)

---
### Kaggle setup
1. Upload the `best_model/` folder as a **Kaggle Dataset** (or use the output of your training notebook).
2. Attach it under *Add-ons → Datasets* → the path will be `/kaggle/input/<dataset-name>/best_model`.
3. Upload your test videos similarly or point `VIDEO_PATHS` to existing Kaggle datasets.

### Colab setup
1. Mount Google Drive (cell 2) and point `MODEL_DIR` to where you saved `best_model/`.
2. Put your test videos somewhere accessible and update `VIDEO_PATHS`.

## 1. Detect Environment & Set Paths

In [ ]:
import os
import sys

# ── Environment detection ───────────────────────────────────────────────────────
ON_KAGGLE = os.path.exists('/kaggle')
ON_COLAB  = 'google.colab' in sys.modules or os.path.exists('/content')

if ON_KAGGLE:
    PLATFORM  = 'kaggle'
    # ⬇ Change <dataset-name> to the name of the Kaggle dataset containing best_model/
    MODEL_DIR = '/kaggle/input/<dataset-name>/best_model'
elif ON_COLAB:
    PLATFORM  = 'colab'
    MODEL_DIR = '/content/drive/MyDrive/TimeSformersModel'
else:
    PLATFORM  = 'local'
    MODEL_DIR = './timesformer_outputs/best_model'

print(f'Platform  : {PLATFORM}')
print(f'Model dir : {MODEL_DIR}')

## 2. (Colab only) Mount Google Drive

Skip on Kaggle or local.

In [ ]:
if PLATFORM == 'colab':
    from google.colab import drive
    drive.mount('/content/drive')
    print('Google Drive mounted.')
else:
    print(f'Skipping Drive mount (platform: {PLATFORM})')

## 3. Install Dependencies

In [ ]:
!pip install -q transformers accelerate safetensors opencv-python-headless pillow torch
print('Dependencies installed.')

## 4. Verify Model Files

Checks that the required files exist in `MODEL_DIR`.  
`preprocessor_config.json` is **optional**: if absent, the processor is fetched from Hugging Face Hub (requires internet).

In [ ]:
# Files that MUST be present
REQUIRED_FILES = ['model.safetensors', 'config.json']
# File that is nice to have but can be fetched from HF Hub as fallback
OPTIONAL_FILES = ['preprocessor_config.json']

print(f'Checking model directory: {MODEL_DIR}\n')
all_ok = True
for fname in REQUIRED_FILES + OPTIONAL_FILES:
    fpath    = os.path.join(MODEL_DIR, fname)
    exists   = os.path.isfile(fpath)
    optional = fname in OPTIONAL_FILES
    if exists:
        size   = f"{os.path.getsize(fpath) / 1024**2:.1f} MB"
        status = '✅'
    elif optional:
        size   = 'not found — will use HF Hub fallback'
        status = '⚠️ '
    else:
        size   = 'MISSING'
        status = '❌'
        all_ok = False
    print(f'  {status}  {fname:35s} {size}')

if not all_ok:
    raise FileNotFoundError(
        f'Required files are missing from {MODEL_DIR}.\n'
        'Make sure you are pointing to the best_model/ directory '
        'produced by Finetuning.py (not a checkpoint-NNNN/ subfolder).'
    )
print('\nCheck complete ✅')

## 5. Load Model & Processor

The processor is loaded from the local checkpoint if `preprocessor_config.json` is present,  
otherwise it falls back to `facebook/timesformer-hr-finetuned-k400` on Hugging Face.  
This is safe because image-preprocessing parameters are not modified during fine-tuning.

In [ ]:
import torch
from transformers import AutoImageProcessor, TimesformerForVideoClassification

# Base model used only as fallback for the processor
BASE_MODEL_ID = 'facebook/timesformer-hr-finetuned-k400'

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Using device: {DEVICE}')

# ── Load processor ────────────────────────────────────────────────────────────
# Try local first; fall back to HF Hub if preprocessor_config.json is absent
_proc_config = os.path.join(MODEL_DIR, 'preprocessor_config.json')
if os.path.isfile(_proc_config):
    processor = AutoImageProcessor.from_pretrained(MODEL_DIR)
    print(f'Processor loaded from local checkpoint.')
else:
    processor = AutoImageProcessor.from_pretrained(BASE_MODEL_ID)
    print(f'preprocessor_config.json not found locally — processor loaded from HF Hub ({BASE_MODEL_ID}).')

# ── Load fine-tuned model ─────────────────────────────────────────────────────
model = TimesformerForVideoClassification.from_pretrained(MODEL_DIR)
model.eval()
model.to(DEVICE)

# Label maps come from the saved config.json automatically
ID2LABEL    = model.config.id2label
LABEL2ID    = model.config.label2id
NUM_CLASSES = model.config.num_labels

print(f'\nModel loaded — {NUM_CLASSES} classes:')
for idx in sorted(ID2LABEL.keys()):
    print(f'  [{idx:2d}] {ID2LABEL[idx]}')

## 6. Video Preprocessing Utility

Samples exactly 16 frames (TimeSformer-HR native count) uniformly from a video file.

In [ ]:
import cv2
import numpy as np
from PIL import Image

NUM_FRAMES = 16   # must match training

def load_video_frames(video_path: str, num_frames: int = NUM_FRAMES) -> list:
    """Sample `num_frames` frames uniformly from a video file.
    Returns a list of PIL Images ready for the processor.
    """
    cap   = cv2.VideoCapture(video_path)
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    if total <= 0:
        total = num_frames  # fallback for unreadable headers

    indices = np.linspace(0, max(total - 1, 0), num_frames, dtype=int)
    frames  = []
    for idx in indices:
        cap.set(cv2.CAP_PROP_POS_FRAMES, int(idx))
        ret, frame = cap.read()
        if ret:
            frames.append(Image.fromarray(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)))
    cap.release()

    # Pad with duplicates if any reads failed
    if not frames:
        frames = [Image.new('RGB', (448, 448))] * num_frames
    while len(frames) < num_frames:
        frames.append(frames[-1])
    return frames[:num_frames]

print('load_video_frames() ready.')

## 7. Inference Function

In [ ]:
import torch.nn.functional as F

@torch.no_grad()
def predict_video(video_path: str, top_k: int = 3) -> dict:
    """Run inference on a single video.

    Args:
        video_path: Path to the .mp4 / .avi / .mov file.
        top_k:      Number of top predictions to return.

    Returns:
        dict with keys:
          'predicted_class'  — top-1 class name (str)
          'predicted_id'     — top-1 class index (int)
          'confidence'       — top-1 softmax score (float)
          'top_k'            — list of (class_name, score) for top-k
    """
    frames = load_video_frames(video_path)
    inputs = processor(images=frames, return_tensors='pt')
    inputs = {k: v.to(DEVICE) for k, v in inputs.items()}

    outputs = model(**inputs)
    logits  = outputs.logits           # (1, num_classes)
    probs   = F.softmax(logits, dim=-1).squeeze(0).cpu().tolist()

    top_indices = sorted(range(len(probs)), key=lambda i: probs[i], reverse=True)[:top_k]
    top_preds   = [(ID2LABEL[i], round(probs[i], 4)) for i in top_indices]

    best_id = top_indices[0]
    return {
        'predicted_class': ID2LABEL[best_id],
        'predicted_id':    best_id,
        'confidence':      round(probs[best_id], 4),
        'top_k':           top_preds,
    }

print('predict_video() ready.')

## 8. Run Inference

Edit `VIDEO_PATHS` to point to your test videos.

Each entry can be:
- An absolute path: `'/kaggle/input/my-videos/clip01.mp4'`
- A relative path from the current working directory

In [ ]:
# ── USER CONFIGURATION ─────────────────────────────────────────────────────────
VIDEO_PATHS = [
    # Add your video paths here, e.g.:
    # '/kaggle/input/my-test-videos/clip_001.mp4',
    # '/content/drive/MyDrive/test_videos/clip_002.mp4',
]
# ──────────────────────────────────────────────────────────────────────────────

if not VIDEO_PATHS:
    print('⚠ VIDEO_PATHS is empty. Add video file paths above and re-run.')
else:
    for vpath in VIDEO_PATHS:
        if not os.path.isfile(vpath):
            print(f'⚠  File not found: {vpath}')
            continue

        result = predict_video(vpath, top_k=3)
        print(f'\nVideo : {os.path.basename(vpath)}')
        print(f'  Predicted : {result["predicted_class"]}  ({result["confidence"]*100:.1f}%)')
        print(f'  Top-3     :')
        for cls, score in result['top_k']:
            bar = '█' * int(score * 30)
            print(f'    {cls:25s} {score*100:5.1f}%  {bar}')

## 9. Batch Inference on a Folder

Scan a directory recursively and run inference on every video found.  
Results are saved to a CSV file.

In [ ]:
import csv
from pathlib import Path

# ── USER CONFIGURATION ─────────────────────────────────────────────────────────
VIDEO_FOLDER = ''           # e.g. '/kaggle/input/test-clips'
OUTPUT_CSV   = 'predictions.csv'
VIDEO_EXTS   = {'.mp4', '.avi', '.mov'}
# ──────────────────────────────────────────────────────────────────────────────

if not VIDEO_FOLDER:
    print('⚠ Set VIDEO_FOLDER to a directory containing videos and re-run.')
else:
    video_files = [
        str(p) for p in Path(VIDEO_FOLDER).rglob('*')
        if p.suffix.lower() in VIDEO_EXTS
    ]
    print(f'Found {len(video_files)} video(s) in {VIDEO_FOLDER}\n')

    rows = []
    for i, vpath in enumerate(sorted(video_files), 1):
        try:
            result = predict_video(vpath, top_k=3)
            row = {
                'file':            os.path.basename(vpath),
                'path':            vpath,
                'predicted_class': result['predicted_class'],
                'confidence':      result['confidence'],
                'top2_class':      result['top_k'][1][0] if len(result['top_k']) > 1 else '',
                'top2_score':      result['top_k'][1][1] if len(result['top_k']) > 1 else '',
                'top3_class':      result['top_k'][2][0] if len(result['top_k']) > 2 else '',
                'top3_score':      result['top_k'][2][1] if len(result['top_k']) > 2 else '',
            }
            rows.append(row)
            print(f'[{i:4d}/{len(video_files)}] {row["file"]:40s}  → {row["predicted_class"]} ({row["confidence"]*100:.1f}%)')
        except Exception as e:
            print(f'[{i:4d}/{len(video_files)}] ERROR on {vpath}: {e}')

    if rows:
        with open(OUTPUT_CSV, 'w', newline='') as f:
            writer = csv.DictWriter(f, fieldnames=rows[0].keys())
            writer.writeheader()
            writer.writerows(rows)
        print(f'\nResults saved to {OUTPUT_CSV}')

## 10. (Optional) Visualise Predictions

Display sampled frames from a video alongside the predicted class probabilities.

In [ ]:
import matplotlib.pyplot as plt

def visualise_prediction(video_path: str, top_k: int = 5):
    """Show a grid of sampled frames and a bar chart of top-k class probabilities."""
    frames = load_video_frames(video_path)
    result = predict_video(video_path, top_k=top_k)

    fig = plt.figure(figsize=(18, 6))
    gs  = fig.add_gridspec(2, 8, hspace=0.4, wspace=0.3)

    # ── Frame grid (top row) ───────────────────────────────────────────────────
    display_frames = frames[::max(1, len(frames) // 8)][:8]   # show up to 8
    for i, frame in enumerate(display_frames):
        ax = fig.add_subplot(gs[0, i])
        ax.imshow(frame)
        ax.set_title(f'f{i+1}', fontsize=8)
        ax.axis('off')

    # ── Bar chart (bottom row, full width) ────────────────────────────────────
    ax_bar  = fig.add_subplot(gs[1, :])
    classes = [c for c, _ in result['top_k']]
    scores  = [s for _, s in result['top_k']]
    colors  = ['#4CAF50' if c == result['predicted_class'] else '#90CAF9' for c in classes]
    bars    = ax_bar.barh(classes[::-1], scores[::-1], color=colors[::-1])
    ax_bar.set_xlim(0, 1)
    ax_bar.set_xlabel('Probability')
    ax_bar.set_title(
        f'Prediction: {result["predicted_class"]}  ({result["confidence"]*100:.1f}%)\n'
        f'File: {os.path.basename(video_path)}'
    )
    for bar, score in zip(bars[::-1], scores):
        ax_bar.text(bar.get_width() + 0.01, bar.get_y() + bar.get_height() / 2,
                    f'{score*100:.1f}%', va='center', fontsize=9)

    plt.show()

# ── Example usage ─────────────────────────────────────────────────────────────
# visualise_prediction('/path/to/your/video.mp4')
print('visualise_prediction() ready.  Uncomment the last line to use it.')

## 11. Annotate Video with Sliding-Window Predictions

Splits the video into segments of `PREDICTION_INTERVAL_SEC` seconds.  
For each segment, **16 frames** are sampled uniformly and passed to the model.  
The resulting label and confidence bar are overlaid on every frame of that segment,  
and the full annotated video is saved to `OUTPUT_VIDEO`.

> The script is located at `annotate_video.py` in the repo root and can also be run from the command line:
> ```
> python annotate_video.py --model_dir ./best_model --input clip.mp4 --interval 1.0
> ```

In [ ]:
import sys

# ── USER CONFIGURATION ─────────────────────────────────────────────────────────
INPUT_VIDEO  = ""   # e.g. "/kaggle/input/my-clips/drive_clip.mp4"
OUTPUT_VIDEO = ""   # leave empty to auto-name next to the input file

# How often the prediction label is updated (seconds)
PREDICTION_INTERVAL_SEC = 1.0

# Frames sampled per segment — must match training (16 for TimeSformer-HR)
FRAMES_PER_SEGMENT = 16
# ──────────────────────────────────────────────────────────────────────────────

if not INPUT_VIDEO:
    print("⚠ Set INPUT_VIDEO to a video file path and re-run.")
else:
    # Make sure annotate_video.py is importable from the repo root
    repo_root = os.path.abspath(os.path.join(os.getcwd(), ".."))
    if repo_root not in sys.path:
        sys.path.insert(0, repo_root)

    from annotate_video import annotate_video

    saved_path = annotate_video(
        model_dir    = MODEL_DIR,
        input_path   = INPUT_VIDEO,
        output_path  = OUTPUT_VIDEO,
        interval_sec = PREDICTION_INTERVAL_SEC,
        num_frames   = FRAMES_PER_SEGMENT,
    )
    print(f"\nDone! Annotated video: {saved_path}")